# Tool, agenti e RAG agentico con datapizza-ai

Questo notebook guida passo passo nella comprensione e implementazione di:

1. **Tool**: funzioni che un LLM può invocare
2. **Agenti**: LLM che usano tool per risolvere problemi
3. **RAG agentico**: sistema RAG dove l'agente decide autonomamente quando e come cercare informazioni

Useremo il framework **datapizza-ai** che semplifica notevolmente la creazione di tool e agenti.

## Prerequisiti

- API key OpenAI configurata nel file `.env`
- Vector store Qdrant popolato (vedi `RAG_Tutorial.ipynb`)


## 1. Setup e configurazione


In [8]:
import os
from dotenv import load_dotenv

# Carica le variabili d'ambiente
load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError("OPENAI_API_KEY non trovata. Crea un file .env con: OPENAI_API_KEY=sk-...")

# Configurazione
LLM_MODEL = "gpt-5.1"
EMBEDDING_MODEL = "text-embedding-3-small"
QDRANT_PATH = "../qdrant_data"
COLLECTION_NAME = "ducati_docs"

print("Setup completato")
print(f"Modello LLM: {LLM_MODEL}")


Setup completato
Modello LLM: gpt-5.1


In [9]:
# Import principali di datapizza-ai
from datapizza.agents import Agent
from datapizza.clients.openai import OpenAIClient
from datapizza.tools import tool

# Crea il client OpenAI che useremo per gli agenti
client = OpenAIClient(api_key=api_key, model=LLM_MODEL)

print("Client OpenAI creato")


Client OpenAI creato


---

## 2. Cos'è un tool?

Un **tool** è una funzione che un LLM può decidere di invocare per ottenere informazioni o eseguire azioni che non può fare da solo.

### Perché servono i tool?

I LLM hanno limitazioni intrinseche:
- Non possono accedere a dati in tempo reale
- Non possono eseguire calcoli complessi in modo affidabile
- Non possono interagire con sistemi esterni

I tool colmano queste lacune permettendo all'LLM di:
- Cercare informazioni aggiornate
- Eseguire operazioni matematiche
- Interrogare database
- Chiamare API esterne

### Come creare un tool con datapizza-ai

Con datapizza-ai, creare un tool è semplicissimo grazie al decoratore `@tool`:

```python
from datapizza.tools import tool

@tool
def nome_funzione(parametro: str) -> str:
    """Descrizione della funzione - l'LLM usa questo per decidere quando usarla."""
    return "risultato"
```

Il decoratore `@tool`:
- Genera automaticamente lo schema JSON dal type hint e dalla docstring
- Registra la funzione come tool utilizzabile dagli agenti


### 2.1 Esempio: tool per il meteo

Creiamo un tool semplice che simula il recupero del meteo di una città.


In [10]:
@tool
def get_weather(city: str, unit: str = "celsius") -> str:
    """
    Restituisce le condizioni meteo attuali di una città italiana.
    
    Args:
        city: Nome della città (es: Bologna, Milano, Roma)
        unit: Unità di misura della temperatura (celsius o fahrenheit)
    """
    # Dati simulati per dimostrazione
    weather_data = {
        "bologna": {"temp": 22, "condition": "soleggiato"},
        "milano": {"temp": 18, "condition": "nuvoloso"},
        "roma": {"temp": 25, "condition": "sereno"},
    }
    
    city_lower = city.lower()
    if city_lower not in weather_data:
        return f"Città {city} non trovata nel database meteo."
    
    data = weather_data[city_lower]
    temp = data["temp"]
    
    if unit == "fahrenheit":
        temp = temp * 9/5 + 32
    
    return f"A {city} ci sono {temp}°{'F' if unit == 'fahrenheit' else 'C'}, cielo {data['condition']}."

# Test della funzione
print(get_weather("Bologna"))
print(get_weather("Milano", "fahrenheit"))


A Bologna ci sono 22°C, cielo soleggiato.
A Milano ci sono 64.4°F, cielo nuvoloso.


### 2.2 Esempio: tool calcolatrice


In [11]:
import math

@tool
def calculate(expression: str) -> str:
    """
    Esegue calcoli matematici. Supporta operazioni base (+, -, *, /), 
    potenze (pow), radici (sqrt), funzioni trigonometriche (sin, cos) e costanti (pi).
    
    Args:
        expression: Espressione matematica da valutare (es: '2 + 2', 'sqrt(16)', 'pow(2, 10)')
    """
    # Operazioni sicure permesse
    allowed_names = {
        "sqrt": math.sqrt,
        "pow": pow,
        "abs": abs,
        "round": round,
        "sin": math.sin,
        "cos": math.cos,
        "pi": math.pi,
    }
    
    try:
        result = eval(expression, {"__builtins__": {}}, allowed_names)
        return f"Il risultato di {expression} è {result}"
    except Exception as e:
        return f"Errore nel calcolo: {str(e)}"

# Test
print(calculate("2 + 2"))
print(calculate("sqrt(144)"))
print(calculate("pow(2, 10)"))


Il risultato di 2 + 2 è 4
Il risultato di sqrt(144) è 12.0
Il risultato di pow(2, 10) è 1024


---

## 3. Cos'è un agente?

Un **agente** è un sistema che usa un LLM come "cervello" per:
1. **Ragionare** sul problema da risolvere
2. **Pianificare** i passi necessari
3. **Agire** usando i tool disponibili
4. **Osservare** i risultati
5. **Iterare** fino a raggiungere l'obiettivo

### Differenza tra LLM con tool e agente

| LLM con tool | Agente |
|--------------|--------|
| Esegue una singola chiamata | Esegue un loop di ragionamento |
| Usa i tool una volta | Può usare tool multipli in sequenza |
| Non ha memoria del contesto | Mantiene lo stato della conversazione |
| Risponde e termina | Continua finché l'obiettivo è raggiunto |

### La classe Agent di datapizza-ai

Con datapizza-ai, creare un agente è molto semplice:

```python
from datapizza.agents import Agent
from datapizza.clients.openai import OpenAIClient

agent = Agent(
    name="nome_agente",
    system_prompt="Istruzioni per l'agente",
    client=OpenAIClient(api_key="...", model="gpt-4o-mini"),
    tools=[tool1, tool2],  # Lista di tool disponibili
    max_steps=10,          # Limite iterazioni
)

response = agent.run("Domanda dell'utente")
print(response.text)
```


### 3.1 Creazione di un agente semplice


In [12]:
# Crea un agente con i tool meteo e calcolatrice
weather_agent = Agent(
    name="weather_calculator_agent",
    system_prompt="""Sei un assistente utile che può:
- Fornire informazioni meteo sulle città italiane
- Eseguire calcoli matematici

Usa i tool disponibili quando necessario. Rispondi sempre in italiano.""",
    client=client,
    tools=[get_weather, calculate],
    max_steps=5,
)

print("Agente creato con tool: get_weather, calculate")


Agente creato con tool: get_weather, calculate


In [13]:
# Test: domanda sul meteo
response = weather_agent.run("Che tempo fa a Bologna?")
print("Risposta:")
print(response.text)


2025-12-04 12:29:07 <weather_calculator_agent> STARTING AGENT 

2025-12-04 12:29:07 <weather_calculator_agent> --- STEP 1 --- 

<weather_calculator_agent>
╭──────────────────────────────────────────── TOOL GET_WEATHER RESULT ────────────────────────────────────────────╮
│ A Bologna ci sono 22°C, cielo soleggiato.                                                                       │
╰─ args: {'city': 'Bologna', 'unit': 'celsius'} ──────────────────────────────────────────────────────────────────╯

2025-12-04 12:29:09 <weather_calculator_agent> --- STEP 2 --- 

<weather_calculator_agent>
╭───────────────────────────────────────────────── FINAL ANSWER ──────────────────────────────────────────────────╮
│ A Bologna in questo momento ci sono 22°C e il cielo è soleggiato.                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Risposta:
A Bologna in questo momento ci sono 22°C e il cielo è soleggiato.


In [14]:
# Test: domanda con calcolo
response = weather_agent.run("Quanto fa 2 elevato alla 16?")
print("Risposta:")
print(response.text)


2025-12-04 12:29:11 <weather_calculator_agent> STARTING AGENT 

2025-12-04 12:29:11 <weather_calculator_agent> --- STEP 1 --- 

<weather_calculator_agent>
╭───────────────────────────────────────────── TOOL CALCULATE RESULT ─────────────────────────────────────────────╮
│ Il risultato di pow(2,16) è 65536                                                                               │
╰─ args: {'expression': 'pow(2,16)'} ─────────────────────────────────────────────────────────────────────────────╯

2025-12-04 12:29:13 <weather_calculator_agent> --- STEP 2 --- 

<weather_calculator_agent>
╭───────────────────────────────────────────────── FINAL ANSWER ──────────────────────────────────────────────────╮
│ 2 elevato alla 16 fa 65.536.                                                                                    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Risposta:
2 elevato alla 16 fa 65.536.


In [15]:
# Test: domanda che richiede ragionamento multi-step
response = weather_agent.run("Se a Bologna ci sono 22 gradi, quanto fa il quadrato della temperatura?")
print("Risposta:")
print(response.text)


2025-12-04 12:29:15 <weather_calculator_agent> STARTING AGENT 

2025-12-04 12:29:15 <weather_calculator_agent> --- STEP 1 --- 

<weather_calculator_agent>
╭───────────────────────────────────────────── TOOL CALCULATE RESULT ─────────────────────────────────────────────╮
│ Il risultato di 22^2 è 20                                                                                       │
╰─ args: {'expression': '22^2'} ──────────────────────────────────────────────────────────────────────────────────╯

2025-12-04 12:29:17 <weather_calculator_agent> --- STEP 2 --- 

<weather_calculator_agent>
╭───────────────────────────────────────────────── FINAL ANSWER ──────────────────────────────────────────────────╮
│ Il quadrato di 22 gradi è 484 (non 20).                                                                         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Risposta:
Il quadrato di 22 gradi è 484 (non 20).


In [16]:
# Test: domanda generica (non richiede tool)
response = weather_agent.run("Ciao, come stai?")
print("Risposta:")
print(response.text)


2025-12-04 12:29:18 <weather_calculator_agent> STARTING AGENT 

2025-12-04 12:29:18 <weather_calculator_agent> --- STEP 1 --- 

<weather_calculator_agent>
╭───────────────────────────────────────────────── FINAL ANSWER ──────────────────────────────────────────────────╮
│ Ciao! Io tutto bene, grazie.                                                                                    │
│ Come posso aiutarti oggi?                                                                                       │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Risposta:
Ciao! Io tutto bene, grazie.  
Come posso aiutarti oggi?


### 3.2 Visualizzare i passi dell'agente con stream_invoke

Il metodo `stream_invoke` permette di vedere ogni passo del ragionamento dell'agente.


In [17]:
# Usa stream_invoke per vedere i passi intermedi
print("Esecuzione con stream_invoke:")
print("="*50)

for step in weather_agent.stream_invoke("Che temperatura c'è a Roma in Fahrenheit?"):
    print(f"\n--- Step {step.index} ---")
    print(f"Testo: {step.text}")


Esecuzione con stream_invoke:


2025-12-04 12:29:20 <weather_calculator_agent> STARTING AGENT 

2025-12-04 12:29:20 <weather_calculator_agent> --- STEP 1 --- 

<weather_calculator_agent>
╭──────────────────────────────────────────── TOOL GET_WEATHER RESULT ────────────────────────────────────────────╮
│ A Roma ci sono 77.0°F, cielo sereno.                                                                            │
╰─ args: {'city': 'Roma', 'unit': 'fahrenheit'} ──────────────────────────────────────────────────────────────────╯


--- Step 1 ---
Testo: 


2025-12-04 12:29:22 <weather_calculator_agent> --- STEP 2 --- 


--- Step 2 ---
Testo: A Roma in questo momento ci sono 77.0°F con cielo sereno.


<weather_calculator_agent>
╭───────────────────────────────────────────────── FINAL ANSWER ──────────────────────────────────────────────────╮
│ A Roma in questo momento ci sono 77.0°F con cielo sereno.                                                       │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

---

## 4. RAG agentico

### Cos'è un RAG agentico?

Un **RAG agentico** è un sistema dove l'agente decide autonomamente:
- **Se** cercare informazioni nel database
- **Cosa** cercare (può riformulare la query)
- **Quante volte** cercare (può fare ricerche multiple)
- **Come** combinare le informazioni trovate

### Differenze tra RAG normale e RAG agentico

| RAG normale | RAG agentico |
|-------------|---------------|
| Cerca sempre | Cerca solo se necessario |
| Una sola ricerca | Ricerche multiple se serve |
| Query utente diretta | Può riformulare la query |
| Flusso fisso | Flusso dinamico |
| Nessun ragionamento | Ragiona su cosa cercare |

### Architettura

```
RAG NORMALE:
  Query → Embedding → Search → Chunks → LLM → Risposta

RAG AGENTICO:
  Query → Agente → [Decide se cercare]
                 → [Tool: search_documents]
                 → [Analizza risultati]
                 → [Decide se cercare ancora]
                 → [Risposta finale]
```

### Vantaggi del RAG agentico

1. **Efficienza**: non cerca se non serve (es: "Ciao, come stai?")
2. **Precisione**: può riformulare query ambigue
3. **Completezza**: può fare ricerche multiple per domande complesse
4. **Flessibilità**: può combinare ricerca con altri tool


### 4.1 Setup del vector store e embedder


In [18]:
from datapizza.vectorstores.qdrant import QdrantVectorstore
from datapizza.embedders.openai import OpenAIEmbedder

# Inizializza il vector store (deve essere già popolato con RAG_Tutorial.ipynb)
vectorstore = QdrantVectorstore(location=None, path=QDRANT_PATH)

# Inizializza l'embedder
embedder = OpenAIEmbedder(
    api_key=api_key,
    model_name=EMBEDDING_MODEL
)

print(f"Vector store caricato da: {QDRANT_PATH}")


Vector store caricato da: ../qdrant_data


### 4.2 Creazione del tool di ricerca documenti


In [19]:
@tool
def search_documents(query: str, num_results: int = 3) -> str:
    """
    Cerca informazioni nei documenti aziendali Ducati indicizzati.
    Usa questo tool per rispondere a domande su Ducati, l'organizzazione, 
    i processi aziendali, i team, le persone e le tecnologie usate.
    
    Args:
        query: Testo da cercare nei documenti. Usa parole chiave specifiche.
        num_results: Numero di risultati da restituire (default: 3)
    """
    try:
        # Genera embedding della query
        query_embedding = embedder.embed(query)
        
        # Cerca nel vector store
        results = vectorstore.search(
            query_vector=query_embedding,
            collection_name=COLLECTION_NAME,
            k=num_results
        )
        
        if not results:
            return "Nessun documento trovato per questa ricerca."
        
        # Formatta i risultati
        formatted_results = []
        for i, chunk in enumerate(results, 1):
            text = chunk.text[:500] + "..." if len(chunk.text) > 500 else chunk.text
            formatted_results.append(f"[Risultato {i}]\n{text}")
        
        return "\n\n".join(formatted_results)
        
    except Exception as e:
        return f"Errore nella ricerca: {str(e)}"

print("Tool search_documents creato")


Tool search_documents creato


In [20]:
# Test del tool di ricerca
print("Test ricerca 'team IT Ducati':")
print(search_documents("team IT Ducati"))


Test ricerca 'team IT Ducati':
[Risultato 1]
Luogo: Borgo Panigale Dipendenti IT: ~40 Consulenti gestiti: ~300 Nota: anche più tecnici sono grandi Project Manager che coordinano team esterni. 2 Persone Chiave Andrea Spina CIO, responsabile R&D; e dati. In Ducati da 20 anni. Spinge sulla cultura Al e sulla trasformazione digitale. Massimiliano Bertei - IT Manager; area Technology in R&D.; Si occupa di sistemi Al per analizzare vantaggi competitivi. Emanuele Arcelli Senior Data Analyst; la Data Platform e il filone AI. 3. Struttura Organizza...

[Risultato 2]
- Supervisione; policy; processi; standardizzazione.
 - Strumenti Al in Ducati
 - Copilot Chat (web) per tutti.
 - Copilot Pro (15 persone) .
 - Copilot Studio (5 persone) .
 - Cultura e Formazione AI



[Risultato 3]
1. Contesto Generale Età media team IT: 45 anni


### 4.3 Creazione dell'agente RAG


In [21]:
# Crea l'agente RAG
rag_agent = Agent(
    name="ducati_rag_agent",
    system_prompt="""Sei un assistente esperto dei documenti aziendali Ducati.

Hai accesso al tool search_documents per cercare informazioni nei documenti indicizzati.

REGOLE IMPORTANTI:
1. Usa search_documents SOLO per domande che riguardano Ducati, l'organizzazione, i team, le persone, i processi o le tecnologie aziendali
2. Per domande generiche (saluti, domande personali, curiosità generali) rispondi direttamente SENZA usare il tool
3. Se la prima ricerca non trova informazioni sufficienti, prova a riformulare la query con parole chiave diverse
4. Basa le tue risposte ESCLUSIVAMENTE sui documenti trovati
5. Se non trovi informazioni nei documenti, dillo chiaramente: "Non ho trovato questa informazione nei documenti disponibili"

Rispondi sempre in italiano in modo chiaro e conciso.""",
    client=client,
    tools=[search_documents],
    max_steps=5,
)

print("Agente RAG creato")


Agente RAG creato


### 4.4 Test dell'agente RAG


In [22]:
# Test 1: Domanda che richiede ricerca nei documenti
print("Domanda: Quanti dipendenti IT ha Ducati?")
print("="*50)

for step in rag_agent.stream_invoke("Quanti dipendenti IT ha Ducati?"):
    print(f"Step {step.index}: {step.text[:200]}..." if len(step.text) > 200 else f"Step {step.index}: {step.text}")


Domanda: Quanti dipendenti IT ha Ducati?


2025-12-04 12:29:25 <ducati_rag_agent> STARTING AGENT 

2025-12-04 12:29:25 <ducati_rag_agent> --- STEP 1 --- 

<ducati_rag_agent>
╭───────────────────────────────────────── TOOL SEARCH_DOCUMENTS RESULT ──────────────────────────────────────────╮
│ [Risultato 1]                                                                                                   │
│ Luogo: Borgo Panigale Dipendenti IT: ~40 Consulenti gestiti: ~300 Nota: anche più tecnici sono grandi Project   │
│ Manager che coordinano team esterni. 2 Persone Chiave Andrea Spina CIO, responsabile R&D; e dati. In Ducati da  │
│ 20 anni. Spinge sulla cultura Al e sulla trasformazione digitale. Massimiliano Bertei - IT Manager; area        │
│ Technology in R&D.; Si occupa di sistemi Al per analizzare vantaggi competitivi. Emanuele Arcelli Senior Data   │
│ Analyst; la Data Platform e il filone AI. 3. Struttura Organizza...                                             │
│                                                                                                                 │
│ [Risultato 2]                                                                                                   │
│ - Supervisione; policy; processi; standardizzazione.                                                            │
│  - Strumenti Al in Ducati                                                                                       │
│  - Copilot Chat (web) per tutti.                                                                                │
│  - Copilot Pro (15 persone) .                                                                                   │
│  - Copilot Studio (5 persone) .                                                                                 │
│  - Cultura e Formazione AI                                                                                      │
│                                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
│ [Risultato 3]                                                                                                   │
│ 1. Contesto Generale Età media team IT: 45 anni                                                                 │
╰─ args: {'query': 'numero dipendenti IT Ducati organico ICT', 'num_results': 3} ─────────────────────────────────╯

Step 1: 


2025-12-04 12:29:28 <ducati_rag_agent> --- STEP 2 --- 

Step 2: Dai documenti disponibili risulta che il team IT di Ducati conta **circa 40 dipendenti interni**, che gestiscono e coordinano **circa 300 consulenti esterni**.  

Se ti serve, posso dettagliare anche ...


<ducati_rag_agent>
╭───────────────────────────────────────────────── FINAL ANSWER ──────────────────────────────────────────────────╮
│ Dai documenti disponibili risulta che il team IT di Ducati conta **circa 40 dipendenti interni**, che           │
│ gestiscono e coordinano **circa 300 consulenti esterni**.                                                       │
│                                                                                                                 │
│ Se ti serve, posso dettagliare anche come è organizzata la struttura IT.                                        │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [23]:
# Test 2: Domanda generica (NON dovrebbe usare il tool)
print("Domanda: Ciao, come ti chiami?")
print("="*50)
response = rag_agent.run("Ciao, come ti chiami?")
print(f"Risposta: {response.text}")


Domanda: Ciao, come ti chiami?


2025-12-04 12:29:30 <ducati_rag_agent> STARTING AGENT 

2025-12-04 12:29:30 <ducati_rag_agent> --- STEP 1 --- 

<ducati_rag_agent>
╭───────────────────────────────────────────────── FINAL ANSWER ──────────────────────────────────────────────────╮
│ Mi chiamo ChatGPT e sono il tuo assistente virtuale Ducati. Come posso aiutarti?                                │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Risposta: Mi chiamo ChatGPT e sono il tuo assistente virtuale Ducati. Come posso aiutarti?


In [24]:
# Test 3: Domanda specifica sui tool AI
print("Domanda: Quali strumenti AI usa Ducati?")
print("="*50)
response = rag_agent.run("Quali strumenti AI usa Ducati?")
print(f"Risposta: {response.text}")


Domanda: Quali strumenti AI usa Ducati?


2025-12-04 12:29:32 <ducati_rag_agent> STARTING AGENT 

2025-12-04 12:29:32 <ducati_rag_agent> --- STEP 1 --- 

<ducati_rag_agent>
╭───────────────────────────────────────── TOOL SEARCH_DOCUMENTS RESULT ──────────────────────────────────────────╮
│ [Risultato 1]                                                                                                   │
│ - Supervisione; policy; processi; standardizzazione.                                                            │
│  - Strumenti Al in Ducati                                                                                       │
│  - Copilot Chat (web) per tutti.                                                                                │
│  - Copilot Pro (15 persone) .                                                                                   │
│  - Copilot Studio (5 persone) .                                                                                 │
│  - Cultura e Formazione AI                                                                                      │
│                                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
│ [Risultato 2]                                                                                                   │
│ Luogo: Borgo Panigale Dipendenti IT: ~40 Consulenti gestiti: ~300 Nota: anche più tecnici sono grandi Project   │
│ Manager che coordinano team esterni. 2 Persone Chiave Andrea Spina CIO, responsabile R&D; e dati. In Ducati da  │
│ 20 anni. Spinge sulla cultura Al e sulla trasformazione digitale. Massimiliano Bertei - IT Manager; area        │
│ Technology in R&D.; Si occupa di sistemi Al per analizzare vantaggi competitivi. Emanuele Arcelli Senior Data   │
│ Analyst; la Data Platform e il filone AI. 3. Struttura Organizza...                                             │
│                                                                                                                 │
│ [Risultato 3]                                                                                                   │
│ - Chatbot; agenti, RAG, multi-agente.                                                                           │
│  - 2x4 ore, due classi: applicativilgovernance e infraldatilsecurity .                                          │
│  - Teoria con minima pratica.                                                                                   │
│  - 6 Obiettivi Specifici                                                                                        │
│  - Diffondere cultura Al e comprensione trasversale.                                                            │
│  - Mappare use case attuali e futuri.                                                                           │
│  - Collegare obiettivi aziendali con opportunità AI.                                                            │
│  - Creare percorsi verticali per PM.                                                                            │
│  - Migliorare comunicazione con team tecnicilpartner .                                                          │
│  - Prossimi Passi                                                                                               │
│  - Workshop Al for Work.                                                                                        │
│  - Sessioni Al for Builders.                                                                                    │
│  - Raccolta feedback e mappa...                                                                                 │
│                                                                                                                 │
│ [Risultato 4]                                                                                                   │
│ - Rimozione timori + 3h di pratica.

2025-12-04 12:29:35 <ducati_rag_agent> --- STEP 2 --- 

<ducati_rag_agent>
╭───────────────────────────────────────────────── FINAL ANSWER ──────────────────────────────────────────────────╮
│ Dai documenti disponibili risulta esplicitamente citato l’uso di questi strumenti AI:                           │
│                                                                                                                 │
│ - **Microsoft Copilot Chat (web)** – reso disponibile a tutte le persone Ducati come assistente generalista.    │
│ - **Microsoft Copilot Pro** – licenze assegnate a circa **15 persone**, tipicamente profili che ne fanno un uso │
│ più intenso/avanzato.                                                                                           │
│ - **Microsoft Copilot Studio** – in uso a circa **5 persone** per creare soluzioni più personalizzate (es.      │
│ chatbot/agent dedicati a processi specifici).                                                                   │
│                                                                                                                 │
│ Nei documenti si parla inoltre in generale di:                                                                  │
│ - **chatbot, agenti, RAG e soluzioni multi‑agente**, ma senza elencare nomi commerciali specifici oltre alla    │
│ suite Copilot.                                                                                                  │
│ - Un filone “Data Platform e AI”, ma senza dettagliare gli strumenti di machine learning usati (es. piattaforme │
│ cloud, framework, ecc.).                                                                                        │
│                                                                                                                 │
│ Oltre a questo, non ho trovato nei documenti elenchi più completi o aggiornati di tutti gli strumenti AI usati  │
│ da Ducati.                                                                                                      │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Risposta: Dai documenti disponibili risulta esplicitamente citato l’uso di questi strumenti AI:

- **Microsoft Copilot Chat (web)** – reso disponibile a tutte le persone Ducati come assistente generalista.
- **Microsoft Copilot Pro** – licenze assegnate a circa **15 persone**, tipicamente profili che ne fanno un uso più intenso/avanzato.
- **Microsoft Copilot Studio** – in uso a circa **5 persone** per creare soluzioni più personalizzate (es. chatbot/agent dedicati a processi specifici).

Nei documenti si parla inoltre in generale di:
- **chatbot, agenti, RAG e soluzioni multi‑agente**, ma senza elencare nomi commerciali specifici oltre alla suite Copilot.
- Un filone “Data Platform e AI”, ma senza dettagliare gli strumenti di machine learning usati (es. piattaforme cloud, framework, ecc.).

Oltre a questo, non ho trovato nei documenti elenchi più completi o aggiornati di tutti gli strumenti AI usati da Ducati.


In [25]:
# Test 4: Domanda su una persona specifica
print("Domanda: Chi è il CIO di Ducati?")
print("="*50)
response = rag_agent.run("Chi è il CIO di Ducati?")
print(f"Risposta: {response.text}")


Domanda: Chi è il CIO di Ducati?


2025-12-04 12:29:41 <ducati_rag_agent> STARTING AGENT 

2025-12-04 12:29:41 <ducati_rag_agent> --- STEP 1 --- 

<ducati_rag_agent>
╭───────────────────────────────────────── TOOL SEARCH_DOCUMENTS RESULT ──────────────────────────────────────────╮
│ [Risultato 1]                                                                                                   │
│ Luogo: Borgo Panigale Dipendenti IT: ~40 Consulenti gestiti: ~300 Nota: anche più tecnici sono grandi Project   │
│ Manager che coordinano team esterni. 2 Persone Chiave Andrea Spina CIO, responsabile R&D; e dati. In Ducati da  │
│ 20 anni. Spinge sulla cultura Al e sulla trasformazione digitale. Massimiliano Bertei - IT Manager; area        │
│ Technology in R&D.; Si occupa di sistemi Al per analizzare vantaggi competitivi. Emanuele Arcelli Senior Data   │
│ Analyst; la Data Platform e il filone AI. 3. Struttura Organizza...                                             │
│                                                                                                                 │
│ [Risultato 2]                                                                                                   │
│ - Supervisione; policy; processi; standardizzazione.                                                            │
│  - Strumenti Al in Ducati                                                                                       │
│  - Copilot Chat (web) per tutti.                                                                                │
│  - Copilot Pro (15 persone) .                                                                                   │
│  - Copilot Studio (5 persone) .                                                                                 │
│  - Cultura e Formazione AI                                                                                      │
│                                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
│ [Risultato 3]                                                                                                   │
│ IT Governance & HR:                                                                                             │
╰─ args: {'query': 'CIO Ducati Chief Information Officer', 'num_results': 3} ─────────────────────────────────────╯

2025-12-04 12:29:42 <ducati_rag_agent> --- STEP 2 --- 

<ducati_rag_agent>
╭───────────────────────────────────────────────── FINAL ANSWER ──────────────────────────────────────────────────╮
│ Dai documenti disponibili risulta che il CIO (Chief Information Officer) di Ducati è **Andrea Spina**.          │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Risposta: Dai documenti disponibili risulta che il CIO (Chief Information Officer) di Ducati è **Andrea Spina**.


In [26]:
# Test 5: Domanda su informazioni probabilmente non presenti
print("Domanda: Qual è il fatturato annuo di Ducati?")
print("="*50)
response = rag_agent.run("Qual è il fatturato annuo di Ducati?")
print(f"Risposta: {response.text}")


Domanda: Qual è il fatturato annuo di Ducati?


2025-12-04 12:29:44 <ducati_rag_agent> STARTING AGENT 

2025-12-04 12:29:44 <ducati_rag_agent> --- STEP 1 --- 

<ducati_rag_agent>
╭───────────────────────────────────────── TOOL SEARCH_DOCUMENTS RESULT ──────────────────────────────────────────╮
│ [Risultato 1]                                                                                                   │
│ Luogo: Borgo Panigale Dipendenti IT: ~40 Consulenti gestiti: ~300 Nota: anche più tecnici sono grandi Project   │
│ Manager che coordinano team esterni. 2 Persone Chiave Andrea Spina CIO, responsabile R&D; e dati. In Ducati da  │
│ 20 anni. Spinge sulla cultura Al e sulla trasformazione digitale. Massimiliano Bertei - IT Manager; area        │
│ Technology in R&D.; Si occupa di sistemi Al per analizzare vantaggi competitivi. Emanuele Arcelli Senior Data   │
│ Analyst; la Data Platform e il filone AI. 3. Struttura Organizza...                                             │
│                                                                                                                 │
│ [Risultato 2]                                                                                                   │
│ - Supervisione; policy; processi; standardizzazione.                                                            │
│  - Strumenti Al in Ducati                                                                                       │
│  - Copilot Chat (web) per tutti.                                                                                │
│  - Copilot Pro (15 persone) .                                                                                   │
│  - Copilot Studio (5 persone) .                                                                                 │
│  - Cultura e Formazione AI                                                                                      │
│                                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
│ [Risultato 3]                                                                                                   │
│ 1. Contesto Generale Età media team IT: 45 anni                                                                 │
╰─ args: {'query': 'fatturato annuo Ducati bilancio ricavi', 'num_results': 3} ───────────────────────────────────╯

2025-12-04 12:29:46 <ducati_rag_agent> --- STEP 2 --- 

<ducati_rag_agent>
╭───────────────────────────────────────────────── FINAL ANSWER ──────────────────────────────────────────────────╮
│ Non ho trovato questa informazione nei documenti disponibili.                                                   │
│                                                                                                                 │
│ Per il dato aggiornato sul fatturato annuo di Ducati ti consiglio di fare riferimento a:                        │
│ - Bilancio annuale Ducati (sezione “Ricavi/Fatturato”), disponibile sul sito ufficiale nella parte investor     │
│ relations o news corporate                                                                                      │
│ - In alternativa, i bilanci del gruppo Volkswagen/Audi, dove Ducati è consolidata, che spesso riportano i       │
│ ricavi per brand.                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Risposta: Non ho trovato questa informazione nei documenti disponibili.

Per il dato aggiornato sul fatturato annuo di Ducati ti consiglio di fare riferimento a:
- Bilancio annuale Ducati (sezione “Ricavi/Fatturato”), disponibile sul sito ufficiale nella parte investor relations o news corporate
- In alternativa, i bilanci del gruppo Volkswagen/Audi, dove Ducati è consolidata, che spesso riportano i ricavi per brand.


### 4.5 Confronto: RAG normale vs RAG agentico

Confrontiamo il comportamento con una domanda generica come "Ciao".


In [27]:
def rag_normale(query: str) -> str:
    """
    RAG classico: cerca SEMPRE nel database e poi genera.
    """
    # 1. Genera embedding della query (SEMPRE)
    query_embedding = embedder.embed(query)
    
    # 2. Cerca nel vector store (SEMPRE, anche per "Ciao")
    results = vectorstore.search(
        query_vector=query_embedding,
        collection_name=COLLECTION_NAME,
        k=3
    )
    
    # 3. Costruisci contesto
    if results:
        context = "\n---\n".join([chunk.text for chunk in results])
    else:
        context = "Nessun documento trovato."
    
    # 4. Genera risposta
    prompt = f"""Basandoti sul contesto, rispondi alla domanda.

CONTESTO:
{context}

DOMANDA: {query}

RISPOSTA:"""
    
    response = client.invoke(prompt)
    return response.text

print("Funzione rag_normale definita")


Funzione rag_normale definita


In [28]:
# Confronto con "Ciao, come stai?"
print("CONFRONTO: 'Ciao, come stai?'")
print("="*60)

print("\n--- RAG NORMALE ---")
print("(Esegue SEMPRE una ricerca nel database, anche per un saluto)")
response_normale = rag_normale("Ciao, come stai?")
print(f"Risposta: {response_normale}")

print("\n--- RAG AGENTICO ---")
print("(Decide se cercare: per un saluto NON cerca)")
response_agentico = rag_agent.run("Ciao, come stai?")
print(f"Risposta: {response_agentico.text}")


CONFRONTO: 'Ciao, come stai?'

--- RAG NORMALE ---
(Esegue SEMPRE una ricerca nel database, anche per un saluto)
Risposta: Ciao, tutto bene, grazie!  
Dal contesto mi sembra che siate in una fase molto interessante di adozione dell’AI in Ducati, soprattutto tra governance, dati e use case applicativi.

Dimmi tu: preferisci parlare dei prossimi passi sulla cultura AI interna, dei workshop (AI for Work / AI for Builders) o vuoi iniziare a mappare dei casi d’uso specifici per l’IT e i PM?

--- RAG AGENTICO ---
(Decide se cercare: per un saluto NON cerca)


2025-12-04 12:29:53 <ducati_rag_agent> STARTING AGENT 

2025-12-04 12:29:53 <ducati_rag_agent> --- STEP 1 --- 

<ducati_rag_agent>
╭───────────────────────────────────────────────── FINAL ANSWER ──────────────────────────────────────────────────╮
│ Ciao! Io non ho stati d’animo, ma sono perfettamente operativo e pronto ad aiutarti 😊                          │
│ Tu come stai? E soprattutto: su cosa ti farebbe comodo un aiuto oggi?                                           │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Risposta: Ciao! Io non ho stati d’animo, ma sono perfettamente operativo e pronto ad aiutarti 😊  
Tu come stai? E soprattutto: su cosa ti farebbe comodo un aiuto oggi?


In [29]:
# Confronto con domanda specifica sui documenti
print("CONFRONTO: 'Chi è Andrea Spina?'")
print("="*60)

print("\n--- RAG NORMALE ---")
response_normale = rag_normale("Chi è Andrea Spina?")
print(f"Risposta: {response_normale}")

print("\n--- RAG AGENTICO ---")
response_agentico = rag_agent.run("Chi è Andrea Spina?")
print(f"Risposta: {response_agentico.text}")


CONFRONTO: 'Chi è Andrea Spina?'

--- RAG NORMALE ---
Risposta: Andrea Spina è il CIO di Ducati, responsabile R&D e dati, in azienda da 20 anni, e guida la spinta sulla cultura AI e sulla trasformazione digitale.

--- RAG AGENTICO ---


2025-12-04 12:29:57 <ducati_rag_agent> STARTING AGENT 

2025-12-04 12:29:57 <ducati_rag_agent> --- STEP 1 --- 

<ducati_rag_agent>
╭───────────────────────────────────────── TOOL SEARCH_DOCUMENTS RESULT ──────────────────────────────────────────╮
│ [Risultato 1]                                                                                                   │
│ Luogo: Borgo Panigale Dipendenti IT: ~40 Consulenti gestiti: ~300 Nota: anche più tecnici sono grandi Project   │
│ Manager che coordinano team esterni. 2 Persone Chiave Andrea Spina CIO, responsabile R&D; e dati. In Ducati da  │
│ 20 anni. Spinge sulla cultura Al e sulla trasformazione digitale. Massimiliano Bertei - IT Manager; area        │
│ Technology in R&D.; Si occupa di sistemi Al per analizzare vantaggi competitivi. Emanuele Arcelli Senior Data   │
│ Analyst; la Data Platform e il filone AI. 3. Struttura Organizza...                                             │
│                                                                                                                 │
│ [Risultato 2]                                                                                                   │
│ 1. Contesto Generale Età media team IT: 45 anni                                                                 │
│                                                                                                                 │
│ [Risultato 3]                                                                                                   │
│ - Supervisione; policy; processi; standardizzazione.                                                            │
│  - Strumenti Al in Ducati                                                                                       │
│  - Copilot Chat (web) per tutti.                                                                                │
│  - Copilot Pro (15 persone) .                                                                                   │
│  - Copilot Studio (5 persone) .                                                                                 │
│  - Cultura e Formazione AI                                                                                      │
│                                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
│ [Risultato 4]                                                                                                   │
│ - Rimozione timori + 3h di pratica.                                                                             │
│  - Focus sul ruolo di copilota dell'AI.                                                                         │
│  - Risultato: portare a casa use case concreto                                                                  │
│                                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
│ [Risultato 5]                                                                                                   │
│ - Chatbot; agenti, RAG, multi-agente.                                                                           │
│  - 2x4 ore, due classi: applicativilgovernance e infraldatilsecurity .                                          │
│  - Teoria con minima pratica.                                                                                   │
│  - 6 Obiettivi Specifici                                                                                        │
│  - Diffondere cultura Al e comprensione trasversale.                                                            │
│  - Mappare use case attuali e futur

2025-12-04 12:29:59 <ducati_rag_agent> --- STEP 2 --- 

<ducati_rag_agent>
╭───────────────────────────────────────────────── FINAL ANSWER ──────────────────────────────────────────────────╮
│ Dai documenti disponibili risulta che:                                                                          │
│                                                                                                                 │
│ Andrea Spina è il CIO di Ducati, responsabile dell’area IT con particolare focus su R&D e dati. È in Ducati da  │
│ circa 20 anni e guida la spinta sulla cultura dell’intelligenza artificiale e sulla trasformazione digitale     │
│ all’interno dell’azienda.                                                                                       │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Risposta: Dai documenti disponibili risulta che:

Andrea Spina è il CIO di Ducati, responsabile dell’area IT con particolare focus su R&D e dati. È in Ducati da circa 20 anni e guida la spinta sulla cultura dell’intelligenza artificiale e sulla trasformazione digitale all’interno dell’azienda.


### 2.1 Esempio: tool per il meteo

Creiamo un tool semplice che simula il recupero del meteo di una città.


In [31]:
# 1. Implementazione della funzione
def get_weather(city: str, unit: str = "celsius") -> dict:
    """
    Restituisce il meteo di una città (dati simulati).
    
    Args:
        city: Nome della città
        unit: Unità di misura (celsius o fahrenheit)
    
    Returns:
        Dizionario con temperatura e condizioni
    """
    # Dati simulati per dimostrazione
    weather_data = {
        "bologna": {"temp": 22, "condition": "soleggiato"},
        "milano": {"temp": 18, "condition": "nuvoloso"},
        "roma": {"temp": 25, "condition": "sereno"},
    }
    
    city_lower = city.lower()
    if city_lower not in weather_data:
        return {"error": f"Città {city} non trovata"}
    
    data = weather_data[city_lower]
    temp = data["temp"]
    
    if unit == "fahrenheit":
        temp = temp * 9/5 + 32
    
    return {
        "city": city,
        "temperature": temp,
        "unit": unit,
        "condition": data["condition"]
    }

# Test della funzione
print(get_weather("Bologna"))
print(get_weather("Milano", "fahrenheit"))


{'city': 'Bologna', 'temperature': 22, 'unit': 'celsius', 'condition': 'soleggiato'}
{'city': 'Milano', 'temperature': 64.4, 'unit': 'fahrenheit', 'condition': 'nuvoloso'}


In [32]:
# 2. Schema JSON del tool per OpenAI
weather_tool = {
    "type": "function",
    "function": {
        "name": "get_weather",
        "description": "Restituisce le condizioni meteo attuali di una città italiana",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {
                    "type": "string",
                    "description": "Nome della città (es: Bologna, Milano, Roma)"
                },
                "unit": {
                    "type": "string",
                    "enum": ["celsius", "fahrenheit"],
                    "description": "Unità di misura della temperatura"
                }
            },
            "required": ["city"]
        }
    }
}

print("Schema del tool:")
print(json.dumps(weather_tool, indent=2))


Schema del tool:
{
  "type": "function",
  "function": {
    "name": "get_weather",
    "description": "Restituisce le condizioni meteo attuali di una citt\u00e0 italiana",
    "parameters": {
      "type": "object",
      "properties": {
        "city": {
          "type": "string",
          "description": "Nome della citt\u00e0 (es: Bologna, Milano, Roma)"
        },
        "unit": {
          "type": "string",
          "enum": [
            "celsius",
            "fahrenheit"
          ],
          "description": "Unit\u00e0 di misura della temperatura"
        }
      },
      "required": [
        "city"
      ]
    }
  }
}


### 2.2 Esempio: tool calcolatrice

Un altro tool comune è una calcolatrice per operazioni matematiche.


In [33]:
import math

def calculate(expression: str) -> dict:
    """
    Valuta un'espressione matematica.
    
    Args:
        expression: Espressione matematica (es: "2 + 2", "sqrt(16)")
    
    Returns:
        Risultato del calcolo
    """
    # Operazioni sicure permesse
    allowed_names = {
        "sqrt": math.sqrt,
        "pow": pow,
        "abs": abs,
        "round": round,
        "sin": math.sin,
        "cos": math.cos,
        "pi": math.pi,
    }
    
    try:
        # Valuta l'espressione in modo sicuro
        result = eval(expression, {"__builtins__": {}}, allowed_names)
        return {"expression": expression, "result": result}
    except Exception as e:
        return {"expression": expression, "error": str(e)}

# Schema del tool
calculator_tool = {
    "type": "function",
    "function": {
        "name": "calculate",
        "description": "Esegue calcoli matematici. Supporta operazioni base (+, -, *, /), potenze (pow), radici (sqrt), funzioni trigonometriche (sin, cos) e costanti (pi)",
        "parameters": {
            "type": "object",
            "properties": {
                "expression": {
                    "type": "string",
                    "description": "Espressione matematica da valutare (es: '2 + 2', 'sqrt(16)', 'pow(2, 10)')"
                }
            },
            "required": ["expression"]
        }
    }
}

# Test
print(calculate("2 + 2"))
print(calculate("sqrt(144)"))
print(calculate("pow(2, 10)"))


{'expression': '2 + 2', 'result': 4}
{'expression': 'sqrt(144)', 'result': 12.0}
{'expression': 'pow(2, 10)', 'result': 1024}


### 2.3 Usare un tool con OpenAI

Vediamo come l'LLM decide se usare un tool e come gestire la risposta.
